In [0]:
%sql
-- 1. A folder for your files
CREATE VOLUME IF NOT EXISTS workspace.bronze.day15_customer_files;

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

customer_schema = StructType([
    StructField("CustomerId", IntegerType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("UpdatedAt", StringType(), True)
])
bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(customer_schema)
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/schema/day15_customers/")
    .load("/Volumes/workspace/bronze/day15_customer_files/")
)
bronze_query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/bronze/checkpoints/day15_bronze/")
    .trigger(availableNow=True)
    .toTable("bronze.day15_customers")
)

bronze_query.awaitTermination()

In [0]:
%sql
SELECT *
FROM bronze.day15_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt,_rescued_data
110,Arun,Chennai,30,2026-08-23 09:00:00,"{""Email"":""arun@gmail.com"",""_file_path"":""/Volumes/workspace/bronze/day15_customer_files/customersData_04.csv""}"
111,Kumar,Bangalore,35,2026-08-23 09:05:00,"{""Email"":""kumar@gmail.com"",""_file_path"":""/Volumes/workspace/bronze/day15_customer_files/customersData_04.csv""}"
112,Ravi,Chennai,null,2026-08-23 10:00:00,"{""Age"":""ABC"",""_file_path"":""/Volumes/workspace/bronze/day15_customer_files/Customers_data_05.csv""}"
113,Raj,Chennai,null,2026-08-23 10:00:00,"{""Age"":""ABC"",""_file_path"":""/Volumes/workspace/bronze/day15_customer_files/Customers_data_06.csv""}"
